In [55]:
from pyspark.sql import SparkSession

### initialize the SparkSession

In [56]:
spark = SparkSession.builder.appName('odais_assignment')\
        .master("local[10]") \
        .getOrCreate()

In [57]:
spark

### define the schema

In [58]:
s = "DEST_COUNTRY_NAME STRING,ORIGIN_COUNTRY_NAME STRING,count INTEGER"

### read the stream of data in csv format

In [59]:
df= spark.readStream.format('csv')\
    .option('path','file:///home/itversity/itversity-material/latest-odai/csv/')\
    .schema(s)\
    .option('header' ,True)\
    .load()

### applay Stateful transformation

In [60]:
from pyspark.sql.functions import sum
df_agg =df.groupby("DEST_COUNTRY_NAME","ORIGIN_COUNTRY_NAME").agg(sum("count"))

In [61]:
spark.conf.set("spark.sql.shuffle.partitions",12)

### write the data into the console

In [62]:
output = df_agg.writeStream.format('console') \
        .outputMode('update') \
        .trigger(processingTime = '5 seconds') \
        .option('truncate', 'false') 

In [63]:
query =output.start()

In [54]:
query.stop()